# Deforestación bruta, recuperación y saldo forestal ponderado en Guatemala

*Resultados nacionales y municipales, 2016–2020*

Juan Alejandro Osorio · IARNA, Universidad Rafael Landívar

Este cuaderno examina qué cambia cuando la recuperación reportada deja de descontarse
de inmediato y en proporción uno a uno de la pérdida bruta. El punto de partida es la
información oficial de la dinámica de cobertura forestal 2016–2020 (INAB & CONAP, 2023).

El recorrido reproduce primero el reporte institucional de pérdida bruta ($B$),
recuperación bruta ($R$) y pérdida neta ($N=B-R$). Después introduce una
*proporción de recuperación de biomasa a veinte años*, $\rho_{20}$, derivada de
evidencia científica publicada, y calcula el saldo forestal ponderado

$$H_i(\rho)=B_i-\rho_iR_i.$$

Los resultados son una aproximación cuantitativa bajo limitaciones explícitas de
datos. No constituyen una cuenta SCAE-CE completa, una medición contemporánea de
biomasa ni una equivalencia ecológica entre pérdida y recuperación.


## Cómo leer el cuaderno

Salvo indicación expresa, las magnitudes corresponden al período de análisis
2016–2020. Una cifra positiva representa pérdida y una negativa, ganancia de cobertura.
El dominio de aplicación comprende 172 municipios asignados mediante
*correspondencia territorial experta codificada*; los otros 168 municipios siguen una regla residual.
Las dos unidades lacustres se conservan en la base como unidades no municipales. El
total nacional se construye después mediante una completación conservadora.

La aproximación de manglar es local y utiliza evidencia estructural de campo. No se
suma a la recuperación ponderada. Los costos de desastres y degradación se mantienen
como contexto no aditivo. El código puede desplegarse, aunque se oculta de inicio para
privilegiar la lectura. Las fórmulas permiten seguir cada cálculo y, después de cada
resultado, una celda Markdown reúne la nota, la fuente y su interpretación.


In [1]:
#@title { display-mode: "form" }
from pathlib import Path
import subprocess
import sys

candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd() / "saldo-forestal-ponderado-guatemala"]
repo = next((
    p.resolve()
    for p in candidatos
    if (p / "04_reproduccion_python" / "src" / "saldo_forestal").is_dir()
), None)
if repo is None:
    destino_repo = Path.cwd() / "saldo-forestal-ponderado-guatemala"
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", "v1.0.0",
            "https://github.com/JA-Osorio/saldo-forestal-ponderado-guatemala.git",
            str(destino_repo),
        ],
        check=True,
    )
    repo = destino_repo.resolve()
sys.path.insert(0, str(repo / "04_reproduccion_python" / "src"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display

from saldo_forestal.reproduccion import ejecutar_reproduccion
from saldo_forestal.visualizacion import (
    config_plotly,
    estilo_plotly,
    mostrar_hallazgo,
    mostrar_tabla,
    mostrar_tarjetas,
    nombre_archivo,
    panel_descargas,
)

def mostrar_figura(fig, titulo, nota, fuente, *, alto=700):
    # Presenta una figura interactiva como una sola salida HTML.
    estilo_plotly(fig, titulo, nota, fuente, alto=alto)
    fragmento = fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config=config_plotly(titulo, alto=int(fig.layout.height or alto)),
        div_id=f"sf_{nombre_archivo(titulo)}",
    )
    display(HTML(fragmento))

productos = ejecutar_reproduccion(repo_dir=repo)
FUENTE_INAB = "INAB y CONAP (2023) e INAB (2023b); cálculos del autor."
FUENTE_RECUPERACION = "INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor."
FUENTE_VALORACION = "Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor."
FUENTE_MANGLE = "INAB (2023a), INAB et al. (2016), e INAB y CONAP (2023); cálculos del autor."
FUENTE_ESCENARIOS = "INAB y CONAP (2023) e INAB (2023b); supuestos y cálculos del autor."


## 1. Antecedente descriptivo del problema de la deforestación neta

La identidad institucional $N=B-R$ es aritméticamente correcta. Su interpretación
como desempeño forestal descuenta toda recuperación de la pérdida en el mismo período.
*Bosques* reúne las estimaciones disponibles desde 1991 y documenta que, entre
1991–2001 y 2010–2016, la pérdida bruta anual aumentó aproximadamente 32 %, mientras
la pérdida neta anual disminuyó cerca de 75 % (Sandoval García et al., 2022).

*El año 1991 aparece porque inicia el primer intervalo recopilado por esa fuente.* No
amplía el período municipal ni entra en los cálculos de 2016–2020. Los intervalos
tienen duraciones y métodos de medición propios; la figura es un antecedente
comparativo y no una serie anual continua.


In [2]:
#@title { display-mode: "form" }
historica = productos["serie_historica"].copy()
larga_historica = historica.melt(
    id_vars="periodo",
    value_vars=["perdida_bruta_anual_reportada_ha", "perdida_neta_anual_reportada_ha"],
    var_name="Indicador",
    value_name="Hectáreas por año",
)
larga_historica["Indicador"] = larga_historica["Indicador"].map({
    "perdida_bruta_anual_reportada_ha": "Pérdida bruta",
    "perdida_neta_anual_reportada_ha": "Pérdida neta",
})
fig_historica = px.line(
    larga_historica,
    x="periodo", y="Hectáreas por año", color="Indicador", symbol="Indicador",
    markers=True,
    color_discrete_map={"Pérdida bruta": "#D55E00", "Pérdida neta": "#0072B2"},
    symbol_map={"Pérdida bruta": "circle", "Pérdida neta": "diamond"},
)
fig_historica.update_traces(marker_size=9, line_width=2.5)
fig_historica.update_xaxes(title="Intervalo de referencia")
fig_historica.update_yaxes(title="ha/año", rangemode="tozero", tickformat=",")

mostrar_figura(
    fig_historica,
    "Figura 1. Divergencia histórica entre pérdida bruta y pérdida neta reportadas",
    "Antecedente descriptivo fuera del análisis municipal 2016–2020. Cada punto corresponde a un intervalo de medición distinto; las líneas facilitan la comparación y no implican continuidad anual.",
    "Sandoval García et al. (2022); cálculos del autor.",
    alto=640,
)


Figura 1. Divergencia histórica entre pérdida bruta y pérdida neta reportadas

*Nota.* Antecedente descriptivo fuera del análisis municipal 2016–2020. Cada punto corresponde a un intervalo de medición distinto; las líneas facilitan la comparación y no implican continuidad anual.

*Fuente.* Sandoval García et al. (2022); cálculos del autor.

La pérdida bruta anual aumenta entre el primer y el último intervalo disponible, mientras la pérdida neta disminuye. La divergencia muestra por qué ambos indicadores deben leerse juntos.


## 2. Reproducción del resultado institucional 2016–2020

Para las 342 unidades de la base oficial —340 municipios y dos unidades lacustres
documentadas por el Instituto Nacional de Bosques (INAB) y el Consejo Nacional de
Áreas Protegidas (CONAP) (2023), con detalle municipal en INAB (2023b)— se reproduce
la identidad:

$$N_i=B_i-R_i.$$

$R_i$ es la recuperación reportada, denominada *ganancia de cobertura forestal* en
la fuente. Se obtiene al comparar las coberturas de 2016 y 2020; no informa la edad,
biomasa, origen o permanencia de esa ganancia.

El total nacional acumulado del período es 244,395 ha de pérdida bruta, 191,658 ha
de recuperación y 52,736 ha de pérdida neta. Esta sección muestra primero el
resultado institucional en sus propios términos y establece el punto de comparación
para la ponderación de la recuperación.


In [3]:
#@title { display-mode: "form" }
nacional = productos["resultados_institucionales_nacionales"].iloc[0]
tabla_nacional = pd.DataFrame({
    "Magnitud": ["Pérdida bruta", "Ganancia de cobertura", "Pérdida neta reportada"],
    "Acumulado 2016–2020 (ha)": [
        nacional.perdida_bruta_ha,
        nacional.recuperacion_bruta_ha,
        nacional.perdida_neta_ha,
    ],
    "Lectura": ["B", "R", "N = B − R"],
})

mostrar_tabla(
    tabla_nacional,
    "Tabla 1. Magnitudes nacionales del cálculo institucional",
    "La recuperación se descuenta en proporción uno a uno; las cifras acumuladas corresponden a 2016–2020.",
    FUENTE_INAB,
    decimales=1,
    max_filas=None,
    archivo="resultados_institucionales_guatemala_2016_2020.csv",
    descarga=productos["resultados_institucionales_nacionales"],
)


Magnitud,Acumulado 2016–2020 (ha),Lectura
Pérdida bruta,"244,394.6",B
Ganancia de cobertura,"191,658.1",R
Pérdida neta reportada,"52,736.4",N = B − R


*Nota.* La recuperación se descuenta en proporción uno a uno; las cifras acumuladas corresponden a 2016–2020.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

La recuperación reportada equivale al 78.4 % de la pérdida bruta. En consecuencia, el resultado neto conserva 52,736 ha, o 21.6 % de la pérdida observada antes de la resta.


In [4]:
#@title { display-mode: "form" }
fig_componentes = go.Figure(go.Waterfall(
    x=["Pérdida bruta", "Ganancia de cobertura", "Pérdida neta"],
    y=[nacional.perdida_bruta_ha, -nacional.recuperacion_bruta_ha, nacional.perdida_neta_ha],
    measure=["absolute", "relative", "total"],
    text=[
        f"+{nacional.perdida_bruta_ha:,.0f}",
        f"−{nacional.recuperacion_bruta_ha:,.0f}",
        f"{nacional.perdida_neta_ha:,.0f}",
    ],
    textposition="outside",
    increasing=dict(marker_color="#D55E00"),
    decreasing=dict(marker_color="#0072B2"),
    totals=dict(marker_color="#009E73"),
    connector=dict(line=dict(color="#8B9AA0", dash="dot")),
    hovertemplate="%{x}<br>%{y:,.1f} ha<extra></extra>",
))
fig_componentes.update_layout(showlegend=False)
fig_componentes.update_yaxes(
    title="ha acumuladas", range=[0, nacional.perdida_bruta_ha * 1.13], tickformat=","
)
fig_componentes.update_xaxes(title=None)

mostrar_figura(
    fig_componentes,
    "Figura 2. Componentes del resultado institucional nacional, 2016–2020",
    "La recuperación se muestra como una sustracción contable. La operación reproduce el reporte institucional, pero no demuestra equivalencia ecológica inmediata.",
    FUENTE_INAB,
    alto=620,
)


Figura 2. Componentes del resultado institucional nacional, 2016–2020

*Nota.* La recuperación se muestra como una sustracción contable. La operación reproduce el reporte institucional, pero no demuestra equivalencia ecológica inmediata.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

La recuperación reduce aritméticamente el resultado nacional de 244,395 a 52,736 ha. La magnitud de esa reducción no implica que la cobertura recuperada tenga la misma condición que la cobertura perdida.


In [5]:
#@title { display-mode: "form" }
departamentales = productos["resultados_institucionales_departamentales"].copy()
tabla_departamental = departamentales[[
    "depto", "perdida_bruta_ha", "recuperacion_bruta_ha", "perdida_neta_ha"
]].rename(columns={
    "depto": "Departamento",
    "perdida_bruta_ha": "Pérdida bruta (ha)",
    "recuperacion_bruta_ha": "Recuperación bruta (ha)",
    "perdida_neta_ha": "Pérdida neta (ha)",
}).sort_values("Pérdida neta (ha)", ascending=False)

mostrar_tabla(
    tabla_departamental,
    "Tabla 2. Resultados departamentales del cálculo institucional",
    "Orden descendente por pérdida neta acumulada. Las dos unidades lacustres permanecen en los agregados departamentales y se identifican en completacion_nacional_unidades.csv dentro de la descarga integral.",
    FUENTE_INAB,
    decimales=1,
    max_filas=None,
    archivo="resultados_institucionales_departamentos_guatemala_2016_2020.csv",
    descarga=departamentales,
)


Departamento,Pérdida bruta (ha),Recuperación bruta (ha),Pérdida neta (ha)
Alta Verapaz,"32,208.6","14,450.7","17,757.9"
Petén,"140,516.8","125,444.3","15,072.5"
Izabal,"16,033.7","5,463.6","10,570.0"
Escuintla,"5,682.7","1,991.8","3,690.9"
Baja Verapaz,"3,815.9",330.4,"3,485.6"
Quiché,"12,843.3","9,622.4","3,220.8"
Huehuetenango,"7,000.7","3,822.3","3,178.4"
Chimaltenango,"4,463.5","1,418.5","3,045.0"
Chiquimula,"2,044.7",141.6,"1,903.1"
Guatemala,"1,744.7",192.0,"1,552.8"


*Nota.* Orden descendente por pérdida neta acumulada. Las dos unidades lacustres permanecen en los agregados departamentales y se identifican en completacion_nacional_unidades.csv dentro de la descarga integral.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

La composición del resultado nacional varía entre departamentos: un mismo saldo neto puede provenir de pérdidas y recuperaciones brutas de magnitudes muy distintas.


In [6]:
#@title { display-mode: "form" }
orden_departamental = departamentales.sort_values("perdida_neta_ha")
fig_departamentos = go.Figure(go.Bar(
    x=orden_departamental["perdida_neta_ha"],
    y=orden_departamental["depto"],
    orientation="h",
    marker_color=np.where(
        orden_departamental["perdida_neta_ha"].ge(0), "#D55E00", "#009E73"
    ),
    text=[f"{v:,.0f}" for v in orden_departamental["perdida_neta_ha"]],
    textposition="outside",
    hovertemplate="%{y}<br>Pérdida neta: %{x:,.1f} ha<extra></extra>",
    name="Pérdida neta",
))
fig_departamentos.add_vline(x=0, line_color="#5C6F77", line_width=1)
minimo_dep = orden_departamental["perdida_neta_ha"].min()
maximo_dep = orden_departamental["perdida_neta_ha"].max()
amplitud_dep = maximo_dep - minimo_dep
fig_departamentos.update_xaxes(
    title="Pérdida neta (ha); la ganancia se muestra a la izquierda",
    range=[min(0, minimo_dep - 0.12 * amplitud_dep), maximo_dep + 0.18 * amplitud_dep],
    tickformat=",",
)
fig_departamentos.update_yaxes(title=None)
mostrar_figura(
    fig_departamentos,
    "Figura 3. Pérdida neta institucional por departamento, 2016–2020",
    "Las barras positivas indican pérdida y las negativas ganancia de cobertura bajo N = B − R.",
    FUENTE_INAB,
    alto=820,
)


Figura 3. Pérdida neta institucional por departamento, 2016–2020

*Nota.* Las barras positivas indican pérdida y las negativas ganancia de cobertura bajo N = B − R.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

La agregación departamental oculta una geografía heterogénea: conviven departamentos con pérdida neta y otros con ganancia de cobertura durante 2016–2020.


In [7]:
#@title { display-mode: "form" }
municipales = productos["resultados_institucionales_municipales"].copy()
extremos_municipales = pd.concat([
    municipales.nsmallest(10, "perdida_neta_ha"),
    municipales.nlargest(10, "perdida_neta_ha"),
]).drop_duplicates("codigo").sort_values("perdida_neta_ha", ascending=False)
tabla_municipal = extremos_municipales[[
    "depto", "municipio", "perdida_neta_ha", "clasificacion_institucional"
]].rename(columns={
    "depto": "Departamento", "municipio": "Municipio",
    "perdida_neta_ha": "Pérdida neta (ha)",
    "clasificacion_institucional": "Clasificación",
})
mostrar_tabla(
    tabla_municipal,
    "Tabla 3. Municipios con mayores pérdidas y ganancias institucionales",
    "Se muestran los diez valores más altos y los diez más bajos; el CSV contiene los 340 municipios.",
    FUENTE_INAB,
    decimales=1,
    max_filas=None,
    archivo="resultados_institucionales_municipios_guatemala_2016_2020.csv",
    descarga=municipales,
)


Departamento,Municipio,Pérdida neta (ha),Clasificación
Petén,San Andrés,"9,166.0",Pérdida
Petén,La Libertad,"8,124.2",Pérdida
Alta Verapaz,Cobán,"5,681.9",Pérdida
Izabal,Livingston,"5,099.6",Pérdida
Alta Verapaz,Senahú,"3,140.2",Pérdida
Izabal,El Estor,"2,855.9",Pérdida
Petén,Poptún,"2,833.9",Pérdida
Escuintla,Escuintla,"2,400.9",Pérdida
Chimaltenango,San Martín Jilotepeque,"2,363.0",Pérdida
Alta Verapaz,San Pedro Carchá,"2,192.1",Pérdida


*Nota.* Se muestran los diez valores más altos y los diez más bajos; el CSV contiene los 340 municipios.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

Los extremos municipales confirman que el resultado nacional no describe una tendencia uniforme. La tabla completa permite examinar los 340 municipios sin reducir el análisis a los casos más visibles.


In [8]:
#@title { display-mode: "form" }
municipales["recuperacion_log1p"] = np.log10(1 + municipales["recuperacion_bruta_ha"])
municipales["perdida_log1p"] = np.log10(1 + municipales["perdida_bruta_ha"])
fig_municipios = px.scatter(
    municipales,
    x="recuperacion_log1p", y="perdida_log1p",
    color="clasificacion_institucional", symbol="clasificacion_institucional",
    hover_name="municipio",
    custom_data=["depto", "recuperacion_bruta_ha", "perdida_bruta_ha", "perdida_neta_ha"],
    labels={
        "recuperacion_log1p": "Ganancia de cobertura (ha; escala log₁₀[1+x])",
        "perdida_log1p": "Pérdida bruta (ha; escala log₁₀[1+x])",
        "clasificacion_institucional": "Clasificación",
    },
    color_discrete_map={"Pérdida": "#D55E00", "Ganancia": "#0072B2", "Equilibrio": "#E69F00"},
    symbol_map={"Pérdida": "circle", "Ganancia": "diamond", "Equilibrio": "square"},
)
limite = max(municipales["recuperacion_bruta_ha"].max(), municipales["perdida_bruta_ha"].max())
fig_municipios.add_trace(go.Scatter(
    x=[0, np.log10(1 + limite)], y=[0, np.log10(1 + limite)], mode="lines", name="B = R",
    line=dict(color="#66777E", dash="dash"), hoverinfo="skip"
))
marcas = [0, 10, 100, 1000, 10000, 30000]
fig_municipios.update_xaxes(tickvals=np.log10(1 + np.array(marcas)), ticktext=[f"{v:,}" for v in marcas])
fig_municipios.update_yaxes(tickvals=np.log10(1 + np.array(marcas)), ticktext=[f"{v:,}" for v in marcas])
fig_municipios.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=8, opacity=0.75, line=dict(width=0.5, color="white")),
    hovertemplate=(
        "%{hovertext}<br>Departamento: %{customdata[0]}"
        "<br>Recuperación: %{customdata[1]:,.1f} ha"
        "<br>Pérdida bruta: %{customdata[2]:,.1f} ha"
        "<br>Pérdida neta: %{customdata[3]:,.1f} ha<extra></extra>"
    ),
)

mostrar_figura(
    fig_municipios,
    "Figura 4. Pérdida bruta y ganancia de cobertura por municipio",
    "La transformación log₁₀(1+x) conserva los 340 municipios, incluidos los valores cero. Sobre B = R hay pérdida institucional; debajo hay ganancia.",
    FUENTE_INAB,
    alto=720,
)


Figura 4. Pérdida bruta y ganancia de cobertura por municipio

*Nota.* La transformación log₁₀(1+x) conserva los 340 municipios, incluidos los valores cero. Sobre B = R hay pérdida institucional; debajo hay ganancia.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); cálculos del autor.

La distancia respecto de la diagonal revela cuánto depende cada resultado municipal de la recuperación. Los municipios próximos a la diagonal son particularmente sensibles al ponderador aplicado.


## 3. Correspondencia territorial e intervalos de recuperación a veinte años

La clasificación se reconstruye como una *correspondencia territorial experta codificada*.
La llave es el código municipal canónico de cuatro dígitos; los nombres
se conservan como etiquetas. Cinco listas explícitas, disjuntas y versionadas asignan
172 municipios a regiones analíticas. Los 168 códigos municipales que no aparecen en
esas listas siguen una regla residual y quedan fuera del dominio de aplicación. Los
dos registros lacustres, sin código municipal, se identifican como unidades no
municipales. La partición completa es, por tanto, $172+168+2=342$ unidades.

Las listas producen exactamente 9 municipios en `REG-PET-N`, 32 en `REG-PET-FTN`,
62 en `REG-TB-HUM`, 35 en `REG-ORI-EST` y 34 en `REG-SEC-MOT`. Cada región se vincula
después con sitios científicos de referencia y con un intervalo de recuperación
relativa de biomasa a veinte años (Poorter et al., 2016, 2017). Esta transferencia
codificada es reproducible; no constituye por sí sola una validación ecológica de las
hectáreas recuperadas en cada municipio.

Para cuatro regiones, los extremos de $\rho_{20}$ son el mínimo y el máximo de los
valores publicados para los sitios seleccionados. En `REG-SEC-MOT`, los sitios con
valor numérico producen $[0.254,0.645]$ y se aplica redondeo exterior a incrementos
de 0.05:

$$\left[0.05\left\lfloor\frac{0.254}{0.05}\right\rfloor,
0.05\left\lceil\frac{0.645}{0.05}\right\rceil\right]=[0.25,0.65].$$

Los intervalos se incorporan al saldo forestal ponderado mediante

$$H_i(\rho_{20})=B_i-\rho_{20,i}R_i.$$

Como $R_i\geq0$, el límite inferior de $H$ usa el límite superior de $\rho_{20}$,
y el límite superior de $H$ usa el límite inferior de $\rho_{20}$:

$$H_i^{\mathrm{inf}}=B_i-\rho_{20,i}^{\mathrm{sup}}R_i,$$

$$H_i^{\mathrm{sup}}=B_i-\rho_{20,i}^{\mathrm{inf}}R_i.$$

$H$ es un *saldo forestal ponderado por recuperación*, no una medición contemporánea
de biomasa ni una corrección oficial de cobertura.


In [9]:
#@title { display-mode: "form" }
catalogo = productos["catalogo_proporciones_recuperacion"].copy()
trazabilidad_territorial = productos["trazabilidad_municipio_region"].copy()
regiones_resumen = productos["resultados_recuperacion_regiones"][[
    "proporcion_region_id", "municipios"
]].copy()
catalogo = catalogo.merge(regiones_resumen, on="proporcion_region_id", how="left")
resumen_trazabilidad = (
    trazabilidad_territorial.groupby(
        ["region_id", "tipo_decision"], dropna=False
    )
    .size()
    .rename("unidades")
    .reset_index()
)
resumen_trazabilidad = resumen_trazabilidad.merge(
    catalogo[["proporcion_region_id", "rho20_min", "rho20_max"]],
    left_on="region_id",
    right_on="proporcion_region_id",
    how="left",
)
resumen_trazabilidad["Intervalo ρ₂₀"] = resumen_trazabilidad.apply(
    lambda f: (
        f"[{f.rho20_min:.3f}, {f.rho20_max:.3f}]"
        if pd.notna(f.rho20_min)
        else "No aplica"
    ),
    axis=1,
)
resumen_trazabilidad["Ruta aplicada"] = resumen_trazabilidad["tipo_decision"].map({
    "lista_explicita": "Lista explícita",
    "regla_residual": "Regla residual",
    "unidad_no_municipal": "Unidad no municipal",
})
orden_regiones = [
    "REG-PET-N", "REG-PET-FTN", "REG-TB-HUM", "REG-ORI-EST",
    "REG-SEC-MOT", "REG-ALT-MON", "UNIDAD-NO-MUN",
]
resumen_trazabilidad["region_id"] = pd.Categorical(
    resumen_trazabilidad["region_id"], categories=orden_regiones, ordered=True
)
tabla_trazabilidad = resumen_trazabilidad[[
    "region_id", "Ruta aplicada", "unidades", "Intervalo ρ₂₀"
]].rename(columns={"region_id": "Región o salida", "unidades": "Unidades"})
tabla_trazabilidad = tabla_trazabilidad.sort_values("Región o salida")

mostrar_tabla(
    tabla_trazabilidad,
    "Tabla 4. Trazabilidad compacta de la correspondencia territorial",
    "Las cinco listas explícitas contienen 172 municipios; la regla residual, 168; y las unidades no municipales, dos. El CSV conserva la regla, el criterio operativo y la fuente para las 342 unidades.",
    "INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.",
    decimales=3,
    max_filas=None,
    archivo="trazabilidad_municipio_region_guatemala_2016_2020.csv",
    descarga=trazabilidad_territorial,
)


Región o salida,Ruta aplicada,Unidades,Intervalo ρ₂₀
REG-PET-N,Lista explícita,9.000,"[0.664, 0.667]"
REG-PET-FTN,Lista explícita,32.000,"[0.594, 0.594]"
REG-TB-HUM,Lista explícita,62.000,"[0.593, 0.766]"
REG-ORI-EST,Lista explícita,35.000,"[0.336, 0.849]"
REG-SEC-MOT,Lista explícita,34.000,"[0.250, 0.650]"
REG-ALT-MON,Regla residual,168.000,No aplica
UNIDAD-NO-MUN,Unidad no municipal,2.000,No aplica


*Nota.* Las cinco listas explícitas contienen 172 municipios; la regla residual, 168; y las unidades no municipales, dos. El CSV conserva la regla, el criterio operativo y la fuente para las 342 unidades.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

La correspondencia territorial queda explícita: 172 municipios se asignan mediante cinco listas de códigos, 168 siguen la regla residual y dos unidades lacustres permanecen fuera del universo municipal.


In [10]:
#@title { display-mode: "form" }
catalogo_fig = catalogo.sort_values("rho20_central")
fig_proporciones = go.Figure(go.Scatter(
    x=catalogo_fig["rho20_central"],
    y=catalogo_fig["region_nombre"],
    mode="markers",
    marker=dict(color="#6A3D9A", size=11, symbol="diamond"),
    error_x=dict(
        type="data",
        array=catalogo_fig["rho20_max"] - catalogo_fig["rho20_central"],
        arrayminus=catalogo_fig["rho20_central"] - catalogo_fig["rho20_min"],
        color="#6A3D9A",
        thickness=2,
        width=7,
    ),
    customdata=catalogo_fig[["sitios_referencia", "municipios"]],
    hovertemplate=(
        "%{y}<br>Proporción central: %{x:.3f}"
        "<br>Sitios: %{customdata[0]}<br>Municipios: %{customdata[1]}<extra></extra>"
    ),
    name="Proporción e intervalo",
))
fig_proporciones.update_xaxes(
    title="Proporción de recuperación de biomasa a veinte años",
    range=[0, 1], tickformat=".0%"
)
fig_proporciones.update_yaxes(title=None)
mostrar_figura(
    fig_proporciones,
    "Figura 5. Intervalos de recuperación de biomasa a veinte años por región de referencia",
    "El punto es la proporción central y la línea su intervalo. Son ponderadores transferidos, no mediciones de edad o biomasa municipal.",
    FUENTE_RECUPERACION,
    alto=650,
)


Figura 5. Intervalos de recuperación de biomasa a veinte años por región de referencia

*Nota.* El punto es la proporción central y la línea su intervalo. Son ponderadores transferidos, no mediciones de edad o biomasa municipal.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

Los intervalos se superponen, pero no son idénticos. La amplitud mostrada se traslada al saldo ponderado y debe interpretarse como variación de las proporciones regionales, no como error muestral.


In [11]:
#@title { display-mode: "form" }
dominio = productos["resultados_recuperacion_dominio"].iloc[0]
tabla_dominio = pd.DataFrame({
    "Magnitud": ["Municipios", "Pérdida bruta", "Pérdida neta", "Saldo ponderado"],
    "Resultado": [
        f"{int(dominio.municipios)}",
        f"{dominio.perdida_bruta_ha:,.0f} ha",
        f"{dominio.perdida_neta_ha:,.0f} ha",
        f"{dominio.saldo_ponderado_inferior_ha:,.0f}–{dominio.saldo_ponderado_superior_ha:,.0f} ha",
    ],
    "Lectura": ["Dominio analítico", "B", "N = B − R", "H, intervalo"],
})

mostrar_tabla(
    tabla_dominio,
    "Tabla 5. Resultado agregado dentro del dominio de 172 municipios",
    "El dominio cubre municipios con una proporción regional defendible; estas cifras no se presentan como total nacional.",
    FUENTE_RECUPERACION,
    decimales=1,
    max_filas=None,
    archivo="resultados_recuperacion_ponderada_dominio_guatemala_2016_2020.csv",
    descarga=productos["resultados_recuperacion_dominio"],
)


Magnitud,Resultado,Lectura
Municipios,172,Dominio analítico
Pérdida bruta,"220,308 ha",B
Pérdida neta,"35,857 ha",N = B − R
Saldo ponderado,"99,593–107,108 ha","H, intervalo"


*Nota.* El dominio cubre municipios con una proporción regional defendible; estas cifras no se presentan como total nacional.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

Dentro de los 172 municipios elegibles, el saldo ponderado asciende a 99,593–107,108 ha, frente a 35,857 ha bajo el cálculo neto reportado.


In [12]:
#@title { display-mode: "form" }
recuperacion_departamentos = productos["resultados_recuperacion_departamentos"].copy()
recuperacion_departamentos["Saldo ponderado (ha)"] = recuperacion_departamentos.apply(
    lambda f: f"{f.saldo_ponderado_inferior_ha:,.0f}–{f.saldo_ponderado_superior_ha:,.0f}",
    axis=1,
)
tabla_recuperacion_departamentos = recuperacion_departamentos[[
    "depto", "municipios", "perdida_neta_ha", "Saldo ponderado (ha)"
]].rename(columns={
    "depto": "Departamento", "municipios": "Municipios del dominio",
    "perdida_neta_ha": "Pérdida neta (ha)",
}).sort_values("Pérdida neta (ha)", ascending=False)

mostrar_tabla(
    tabla_recuperacion_departamentos,
    "Tabla 6. Resultados departamentales dentro del dominio de aplicación",
    "Cada agregado incluye únicamente municipios con proporción regional asignada; por ello no equivale necesariamente al total departamental.",
    FUENTE_RECUPERACION,
    decimales=1,
    max_filas=None,
    archivo="resultados_recuperacion_ponderada_departamentos_guatemala_2016_2020.csv",
    descarga=recuperacion_departamentos,
)


Departamento,Municipios del dominio,Pérdida neta (ha),Saldo ponderado (ha)
Petén,14.0,"15,072.5","60,491–60,717"
Alta Verapaz,11.0,"10,952.5","16,145–16,145"
Izabal,5.0,"10,570.0","11,849–12,794"
Escuintla,12.0,"3,517.8","3,984–4,328"
Baja Verapaz,7.0,"2,648.3","2,753–2,873"
Chiquimula,11.0,"1,903.1","1,925–1,997"
Huehuetenango,10.0,"1,798.5","3,111–3,111"
El Progreso,8.0,"1,108.1","1,258–1,430"
Jalapa,7.0,950.1,"982–1,092"
Guatemala,4.0,208.9,213–218


*Nota.* Cada agregado incluye únicamente municipios con proporción regional asignada; por ello no equivale necesariamente al total departamental.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

El efecto de la ponderación difiere por departamento porque cambian tanto la recuperación observada como la proporción regional asignada a sus municipios elegibles.


In [13]:
#@title { display-mode: "form" }
recuperacion_municipios = productos["resultados_recuperacion_municipios"].copy()
recuperacion_municipios["recuperacion_reconocida_central_ha"] = (
    recuperacion_municipios["rho20_central"] * recuperacion_municipios["recuperacion_bruta_ha"]
)
recuperacion_municipios["clasificacion_central"] = np.select(
    [
        recuperacion_municipios["saldo_ponderado_central_ha"].gt(0),
        recuperacion_municipios["saldo_ponderado_central_ha"].lt(0),
    ],
    ["Pérdida", "Ganancia"],
    default="Equilibrio",
)
panel_institucional = recuperacion_municipios.assign(
    Tratamiento="Cálculo reportado: R completa",
    recuperacion_reconocida_ha=recuperacion_municipios["recuperacion_bruta_ha"],
    Clasificación=recuperacion_municipios["clasificacion_institucional"],
)
panel_ponderado = recuperacion_municipios.assign(
    Tratamiento="Ponderación: ρ central × R",
    recuperacion_reconocida_ha=recuperacion_municipios["recuperacion_reconocida_central_ha"],
    Clasificación=recuperacion_municipios["clasificacion_central"],
)
dispersion_paneles = pd.concat([panel_institucional, panel_ponderado], ignore_index=True)
dispersion_paneles["x_log1p"] = np.log10(1 + dispersion_paneles["recuperacion_reconocida_ha"])
dispersion_paneles["y_log1p"] = np.log10(1 + dispersion_paneles["perdida_bruta_ha"])
fig_paneles = px.scatter(
    dispersion_paneles,
    x="x_log1p", y="y_log1p", facet_col="Tratamiento",
    facet_col_spacing=0.06,
    color="Clasificación", symbol="Clasificación", hover_name="municipio",
    custom_data=["depto", "recuperacion_reconocida_ha", "perdida_bruta_ha"],
    color_discrete_map={"Pérdida": "#D55E00", "Ganancia": "#0072B2", "Equilibrio": "#E69F00"},
    symbol_map={"Pérdida": "circle", "Ganancia": "diamond", "Equilibrio": "square"},
    category_orders={"Tratamiento": ["Cálculo reportado: R completa", "Ponderación: ρ central × R"]},
)
limite_panel = max(
    dispersion_paneles["recuperacion_reconocida_ha"].max(),
    dispersion_paneles["perdida_bruta_ha"].max(),
)
limite_panel_log = np.log10(1 + limite_panel)
for columna in (1, 2):
    fig_paneles.add_shape(
        type="line", x0=0, y0=0, x1=limite_panel_log, y1=limite_panel_log,
        line=dict(color="#5C6F77", dash="dash"), row=1, col=columna,
    )
marcas_panel = [0, 10, 100, 1000, 10000, 30000]
fig_paneles.update_xaxes(
    title=None, matches="x",
    tickvals=np.log10(1 + np.array(marcas_panel)), ticktext=[f"{v:,}" for v in marcas_panel]
)
fig_paneles.update_yaxes(
    title=None, matches="y",
    tickvals=np.log10(1 + np.array(marcas_panel)), ticktext=[f"{v:,}" for v in marcas_panel]
)
fig_paneles.update_yaxes(
    title_text="Pérdida bruta (ha; log₁₀[1+x])", row=1, col=1
)
fig_paneles.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_paneles.add_annotation(
    text="Recuperación reconocida (ha; log₁₀[1+x])",
    x=0.5, y=0, xref="paper", yref="paper", yshift=-44,
    showarrow=False, font=dict(size=12, color="#24363D"),
)
fig_paneles.update_traces(
    marker=dict(size=7, opacity=0.72, line=dict(width=0.4, color="white")),
    hovertemplate=(
        "%{hovertext}<br>Departamento: %{customdata[0]}"
        "<br>Recuperación reconocida: %{customdata[1]:,.1f} ha"
        "<br>Pérdida bruta: %{customdata[2]:,.1f} ha<extra></extra>"
    ),
)
mostrar_figura(
    fig_paneles,
    "Figura 6. Dispersión municipal antes y después de aplicar la ponderación",
    "Los dos paneles usan los mismos 172 municipios y escalas. La diagonal compara B con la recuperación reconocida; el panel ponderado reduce R mediante la proporción central.",
    FUENTE_RECUPERACION,
    alto=770,
)


Figura 6. Dispersión municipal antes y después de aplicar la ponderación

*Nota.* Los dos paneles usan los mismos 172 municipios y escalas. La diagonal compara B con la recuperación reconocida; el panel ponderado reduce R mediante la proporción central.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

Después de ponderar la recuperación, los puntos se desplazan hacia abajo respecto de la comparación original. El cambio permite verificar visualmente que el ajuste no es un multiplicador uniforme del resultado neto.


In [14]:
#@title { display-mode: "form" }
escala_simetica = lambda s: np.sign(s) * np.log10(1 + np.abs(s) / 100)
recuperacion_municipios["N_transformado"] = escala_simetica(recuperacion_municipios["perdida_neta_ha"])
recuperacion_municipios["H_transformado"] = escala_simetica(recuperacion_municipios["saldo_ponderado_central_ha"])
recuperacion_municipios["Estado del cambio"] = np.select(
    [
        recuperacion_municipios["clasificacion_institucional"].eq("Ganancia")
        & recuperacion_municipios["clasificacion_ponderada"].eq("Pérdida"),
        recuperacion_municipios["clasificacion_ponderada"].eq("Indeterminado"),
    ],
    ["Ganancia → pérdida", "Hacia indeterminado"],
    default="Sin cambio de clase",
)
fig_cambio = px.scatter(
    recuperacion_municipios,
    x="N_transformado", y="H_transformado", color="Estado del cambio",
    symbol="Estado del cambio", hover_name="municipio",
    custom_data=[
        "depto", "perdida_neta_ha", "saldo_ponderado_central_ha",
        "saldo_ponderado_inferior_ha", "saldo_ponderado_superior_ha"
    ],
    color_discrete_map={
        "Sin cambio de clase": "#8B9AA0",
        "Ganancia → pérdida": "#D55E00",
        "Hacia indeterminado": "#E69F00",
    },
    symbol_map={
        "Sin cambio de clase": "circle-open",
        "Ganancia → pérdida": "diamond",
        "Hacia indeterminado": "square",
    },
)
limite_cambio = max(
    recuperacion_municipios["N_transformado"].abs().max(),
    recuperacion_municipios["H_transformado"].abs().max(),
)
fig_cambio.add_trace(go.Scatter(
    x=[-limite_cambio, limite_cambio], y=[-limite_cambio, limite_cambio],
    mode="lines", name="H = N", line=dict(color="#5C6F77", dash="dash"),
    hoverinfo="skip",
))
fig_cambio.add_hline(y=0, line_color="#AAB7BC", line_width=1)
fig_cambio.add_vline(x=0, line_color="#AAB7BC", line_width=1)
marcas_simeticas = [-10000, -1000, -100, 0, 100, 1000, 10000]
posiciones_simeticas = escala_simetica(np.array(marcas_simeticas))
fig_cambio.update_xaxes(
    title="Pérdida neta institucional, N (ha; escala simétrica)",
    tickvals=posiciones_simeticas, ticktext=[f"{v:,}" for v in marcas_simeticas]
)
fig_cambio.update_yaxes(
    title="Saldo ponderado central, H (ha; escala simétrica)",
    tickvals=posiciones_simeticas, ticktext=[f"{v:,}" for v in marcas_simeticas]
)
fig_cambio.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=8, opacity=0.8),
    hovertemplate=(
        "%{hovertext}<br>Departamento: %{customdata[0]}"
        "<br>N institucional: %{customdata[1]:,.1f} ha"
        "<br>H central: %{customdata[2]:,.1f} ha"
        "<br>Intervalo H: %{customdata[3]:,.1f}–%{customdata[4]:,.1f} ha<extra></extra>"
    ),
)
mostrar_figura(
    fig_cambio,
    "Figura 7. Cambio municipal entre la pérdida neta y el saldo ponderado",
    "La misma transformación simétrica se aplica a ambos ejes y conserva negativos y ceros. Quince municipios pasan de ganancia a pérdida y once quedan indeterminados o cambian desde equilibrio.",
    FUENTE_RECUPERACION,
    alto=760,
)


Figura 7. Cambio municipal entre la pérdida neta y el saldo ponderado

*Nota.* La misma transformación simétrica se aplica a ambos ejes y conserva negativos y ceros. Quince municipios pasan de ganancia a pérdida y once quedan indeterminados o cambian desde equilibrio.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

La ponderación eleva el saldo de pérdida en la mayor parte del dominio. Veintiséis municipios cambian de clasificación, incluidos quince que pasan de ganancia reportada a pérdida.


In [15]:
#@title { display-mode: "form" }
transiciones = productos["transiciones_clasificacion_ponderada"].copy()
orden_i = ["Ganancia", "Equilibrio", "Pérdida"]
orden_p = ["Ganancia", "Indeterminado", "Pérdida"]
matriz_n = transiciones.pivot(
    index="clasificacion_institucional", columns="clasificacion_ponderada", values="municipios"
).reindex(index=orden_i, columns=orden_p).fillna(0)
matriz_pct = transiciones.pivot(
    index="clasificacion_institucional", columns="clasificacion_ponderada", values="porcentaje_fila"
).reindex(index=orden_i, columns=orden_p).fillna(0)
texto_matriz = [
    [f"{int(matriz_n.iloc[i, j])}<br>{matriz_pct.iloc[i, j]:.1f}%" for j in range(len(orden_p))]
    for i in range(len(orden_i))
]
fig_transiciones = go.Figure(go.Heatmap(
    z=matriz_n.values,
    x=orden_p,
    y=orden_i,
    text=texto_matriz,
    texttemplate="%{text}",
    colorscale=[[0, "#F7FAFA"], [1, "#6A3D9A"]],
    colorbar=dict(title="Municipios"),
    hovertemplate="Institucional: %{y}<br>Ponderada: %{x}<br>Municipios: %{z}<extra></extra>",
))
fig_transiciones.update_xaxes(title="Clasificación con intervalo ponderado", side="bottom")
fig_transiciones.update_yaxes(title="Clasificación institucional", autorange="reversed")
mostrar_figura(
    fig_transiciones,
    "Figura 8. Transición de clasificaciones municipales al aplicar el ponderador",
    "Cada celda muestra municipios y porcentaje dentro de la clasificación institucional de origen. Cambian 26 de 172 diagnósticos.",
    FUENTE_RECUPERACION,
    alto=650,
)


Figura 8. Transición de clasificaciones municipales al aplicar el ponderador

*Nota.* Cada celda muestra municipios y porcentaje dentro de la clasificación institucional de origen. Cambian 26 de 172 diagnósticos.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

El 15.1 % de los municipios del dominio cambia de diagnóstico. La transición se concentra en resultados próximos a cero, donde la resta completa de la recuperación determina el signo.


In [16]:
#@title { display-mode: "form" }
tabla_transiciones = matriz_n.reset_index().rename(columns={
    "clasificacion_institucional": "Clasificación institucional",
    "Ganancia": "Ponderada: ganancia",
    "Indeterminado": "Ponderada: indeterminado",
    "Pérdida": "Ponderada: pérdida",
})
mostrar_tabla(
    tabla_transiciones,
    "Tabla 7. Matriz de transición de clasificaciones municipales",
    "Los conteos corresponden al dominio de 172 municipios; la clasificación ponderada usa el intervalo completo, no solo el punto central.",
    FUENTE_RECUPERACION,
    decimales=0,
    max_filas=None,
    archivo="transiciones_clasificacion_ponderada_municipios_guatemala_2016_2020.csv",
    descarga=transiciones,
)


Clasificación institucional,Ponderada: ganancia,Ponderada: indeterminado,Ponderada: pérdida
Ganancia,42,9,15
Equilibrio,0,2,0
Pérdida,0,0,104


*Nota.* Los conteos corresponden al dominio de 172 municipios; la clasificación ponderada usa el intervalo completo, no solo el punto central.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

La matriz distingue permanencias y transiciones sin perder el denominador de cada grupo de origen. La mayoría conserva su clasificación, pero los cambios no son marginales.


In [17]:
#@title { display-mode: "form" }
cambios = productos["municipios_cambio_clasificacion_ponderada"].copy()
cambios["brecha_central_ha"] = (
    cambios["saldo_ponderado_central_ha"] - cambios["perdida_neta_ha"]
)
cambios_visibles = cambios.nlargest(20, "brecha_central_ha")
tabla_cambios = cambios_visibles[[
    "depto", "municipio", "perdida_neta_ha", "saldo_ponderado_central_ha",
    "cambio_clasificacion"
]].rename(columns={
    "depto": "Departamento", "municipio": "Municipio",
    "perdida_neta_ha": "Pérdida neta (ha)",
    "saldo_ponderado_central_ha": "Saldo ponderado central (ha)",
    "cambio_clasificacion": "Cambio",
})
mostrar_tabla(
    tabla_cambios,
    "Tabla 8. Municipios con mayor cambio de diagnóstico al ponderar la recuperación",
    "Se muestran los veinte mayores aumentos del saldo; el CSV contiene los 26 municipios cuya clasificación cambia con el intervalo ponderado.",
    FUENTE_RECUPERACION,
    decimales=1,
    max_filas=None,
    archivo="cambios_clasificacion_ponderada_municipios_guatemala_2016_2020.csv",
    descarga=cambios,
)


Departamento,Municipio,Pérdida neta (ha),Saldo ponderado central (ha),Cambio
Petén,Sayaxché,"-1,763.5","4,038.0",Ganancia → Pérdida
Petén,San Luis,"-1,018.8","4,409.5",Ganancia → Pérdida
Petén,Dolores,"-3,185.1",994.0,Ganancia → Pérdida
Petén,Flores,-988.8,"1,762.9",Ganancia → Pérdida
Petén,Santa Ana,"-1,626.8",333.6,Ganancia → Pérdida
Quiché,Ixcán,-972.8,811.0,Ganancia → Pérdida
Zacapa,Gualán,-814.9,-27.3,Ganancia → Indeterminado
Quiché,Nebaj,-319.3,205.9,Ganancia → Pérdida
Retalhuleu,San Andrés Villa Seca,-278.4,65.3,Ganancia → Indeterminado
Santa Rosa,Chiquimulilla,-151.8,93.1,Ganancia → Pérdida


*Nota.* Se muestran los veinte mayores aumentos del saldo; el CSV contiene los 26 municipios cuya clasificación cambia con el intervalo ponderado.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

Los mayores cambios se concentran donde la recuperación reportada es alta. La tabla permite identificar los municipios que explican el cambio agregado y no solo su magnitud nacional.


In [18]:
#@title { display-mode: "form" }
forestal_dep = recuperacion_departamentos.copy()
forestal_dep["centro"] = (
    forestal_dep["saldo_ponderado_inferior_ha"] + forestal_dep["saldo_ponderado_superior_ha"]
) / 2
forestal_dep = forestal_dep.sort_values("centro")
fig_departamentos_recuperacion = go.Figure()
for fila in forestal_dep.itertuples(index=False):
    fig_departamentos_recuperacion.add_trace(go.Scatter(
        x=[fila.perdida_neta_ha, fila.centro], y=[fila.depto, fila.depto],
        mode="lines", line=dict(color="#CBD5D8", width=2),
        showlegend=False, hoverinfo="skip",
    ))
fig_departamentos_recuperacion.add_trace(go.Scatter(
    x=forestal_dep["perdida_neta_ha"], y=forestal_dep["depto"],
    mode="markers", name="Pérdida neta institucional",
    marker=dict(color="#0072B2", symbol="square", size=8),
    hovertemplate="%{y}<br>N: %{x:,.1f} ha<extra></extra>",
))
fig_departamentos_recuperacion.add_trace(go.Scatter(
    x=forestal_dep["centro"], y=forestal_dep["depto"],
    mode="markers", name="Saldo ponderado",
    marker=dict(color="#6A3D9A", symbol="diamond", size=9),
    error_x=dict(
        type="data",
        array=forestal_dep["saldo_ponderado_superior_ha"] - forestal_dep["centro"],
        arrayminus=forestal_dep["centro"] - forestal_dep["saldo_ponderado_inferior_ha"],
        color="#6A3D9A",
    ),
    hovertemplate="%{y}<br>H central: %{x:,.1f} ha<extra></extra>",
))
fig_departamentos_recuperacion.add_vline(x=0, line_color="#5C6F77", line_width=1)
fig_departamentos_recuperacion.update_xaxes(title="Resultado (ha); la pérdida es positiva", tickformat=",")
fig_departamentos_recuperacion.update_yaxes(title=None)
mostrar_figura(
    fig_departamentos_recuperacion,
    "Figura 9. Cambio departamental entre pérdida neta y saldo ponderado",
    "Solo incluye municipios del dominio de aplicación. El cuadrado es N; el diamante y su intervalo representan H. Las líneas grises unen lecturas alternativas.",
    FUENTE_RECUPERACION,
    alto=820,
)


Figura 9. Cambio departamental entre pérdida neta y saldo ponderado

*Nota.* Solo incluye municipios del dominio de aplicación. El cuadrado es N; el diamante y su intervalo representan H. Las líneas grises unen lecturas alternativas.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

Todos los departamentos del dominio se desplazan hacia un saldo de mayor pérdida al limitar la recuperación reconocida. La longitud de cada línea resume la magnitud territorial del ajuste.


## 4. Completación conservadora del resultado nacional

Para no extrapolar las proporciones regionales a ecosistemas sin un puente defendible, la
completación nacional aplica $\rho_{20}$ únicamente dentro del dominio de 172
municipios. Fuera de él conserva el cálculo institucional $\rho=1$:

$$H_{GT}=\sum_{i\in P}(B_i-\rho_{20,i}R_i)+\sum_{i\notin P}(B_i-R_i).$$

Esta decisión conserva fuera del dominio la compensación completa de la recuperación.
El resultado nacional ponderado (116,473–123,988 ha) permanece entre la pérdida
bruta y la pérdida neta institucional (INAB & CONAP, 2023; Poorter et al., 2016, 2017).


In [19]:
#@title { display-mode: "form" }
completacion = productos["completacion_nacional_resumen"].copy()
fila_completacion = completacion.iloc[0]
tabla_completacion = pd.DataFrame({
    "Magnitud": [
        "Unidades nacionales", "Municipios con proporción a veinte años",
        "Pérdida bruta", "Ganancia de cobertura", "Pérdida neta",
        "Saldo ponderado nacional",
    ],
    "Resultado": [
        f"{int(fila_completacion.unidades)}",
        f"{int(fila_completacion.municipios_con_proporcion)}",
        f"{fila_completacion.perdida_bruta_ha:,.0f} ha",
        f"{fila_completacion.recuperacion_bruta_ha:,.0f} ha",
        f"{fila_completacion.perdida_neta_ha:,.0f} ha",
        f"{fila_completacion.saldo_ponderado_inferior_ha:,.0f}–{fila_completacion.saldo_ponderado_superior_ha:,.0f} ha",
    ],
    "Método": ["Cobertura", "ρ₂₀", "B", "R", "N = B − R", "Completación conservadora"],
})

mostrar_tabla(
    tabla_completacion,
    "Tabla 9. Completación conservadora del saldo forestal nacional",
    "Dentro del dominio se aplica el intervalo de ρ₂₀; fuera se conserva ρ = 1. El intervalo refleja únicamente las proporciones transferidas.",
    FUENTE_RECUPERACION,
    decimales=1,
    max_filas=None,
    archivo="completacion_conservadora_resumen_guatemala_2016_2020.csv",
    descarga=completacion,
)


Magnitud,Resultado,Método
Unidades nacionales,342,Cobertura
Municipios con proporción a veinte años,172,ρ₂₀
Pérdida bruta,"244,395 ha",B
Ganancia de cobertura,"191,658 ha",R
Pérdida neta,"52,736 ha",N = B − R
Saldo ponderado nacional,"116,473–123,988 ha",Completación conservadora


*Nota.* Dentro del dominio se aplica el intervalo de ρ₂₀; fuera se conserva ρ = 1. El intervalo refleja únicamente las proporciones transferidas.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

La completación conservadora sitúa el saldo nacional entre 116,473 y 123,988 ha: de 2.21 a 2.35 veces la pérdida neta reportada, sin extrapolar las proporciones fuera de su dominio de aplicación.


In [20]:
#@title { display-mode: "form" }
resultados_nacionales = productos["resultados_forestales_nacionales"].copy()
resultados_nacionales["central"] = (
    resultados_nacionales["resultado_inferior_ha"]
    + resultados_nacionales["resultado_superior_ha"]
) / 2
resultados_nacionales["err_mas"] = (
    resultados_nacionales["resultado_superior_ha"] - resultados_nacionales["central"]
)
resultados_nacionales["err_menos"] = (
    resultados_nacionales["central"] - resultados_nacionales["resultado_inferior_ha"]
)
orden_resultados = ["Deforestación bruta", "Saldo ponderado por recuperación", "Pérdida neta institucional"]
resultados_nacionales["regla"] = pd.Categorical(
    resultados_nacionales["regla"], categories=orden_resultados, ordered=True
)
resultados_nacionales = resultados_nacionales.sort_values("regla")
fig_resultados_nacionales = go.Figure(go.Scatter(
    x=resultados_nacionales["central"], y=resultados_nacionales["regla"].astype(str), mode="markers+text",
    marker=dict(
        color=["#D55E00", "#6A3D9A", "#0072B2"],
        symbol=["circle", "diamond", "square"], size=12,
    ),
    error_x=dict(
        type="data",
        array=resultados_nacionales["err_mas"],
        arrayminus=resultados_nacionales["err_menos"],
    ),
    text=[f"  {v:,.0f}" for v in resultados_nacionales["central"]], textposition="middle right",
    hovertemplate="%{y}<br>%{x:,.0f} ha<extra></extra>",
))
maximo_resultados = resultados_nacionales["resultado_superior_ha"].max()
fig_resultados_nacionales.update_xaxes(
    title="ha acumuladas, 2016–2020", range=[0, maximo_resultados * 1.20], tickformat=","
)
fig_resultados_nacionales.update_yaxes(title=None)

mostrar_figura(
    fig_resultados_nacionales,
    "Figura 10. Deforestación bruta, saldo ponderado y pérdida neta nacional",
    "La línea del saldo ponderado muestra el intervalo nacional conservador. Los tres resultados parten de la misma base y no son categorías aditivas.",
    FUENTE_RECUPERACION,
    alto=660,
)


Figura 10. Deforestación bruta, saldo ponderado y pérdida neta nacional

*Nota.* La línea del saldo ponderado muestra el intervalo nacional conservador. Los tres resultados parten de la misma base y no son categorías aditivas.

*Fuente.* INAB y CONAP (2023), INAB (2023b) y Poorter et al. (2016, 2017); cálculos del autor.

El saldo ponderado queda entre la pérdida bruta y la pérdida neta. La separación entre los tres resultados cuantifica cuánto cambia la lectura nacional según el reconocimiento otorgado a la recuperación.


## 5. Valoración indicativa de los resultados forestales

La transferencia utiliza Q29,986 por ha por año, valor homologado a 2026 a partir de
Q22,553 por ha por año. La Cuenta de ecosistemas de Guatemala integra 21 estudios
sobre 9,403 km², cerca de 8 % del territorio, y combina servicios y métodos distintos
(Banco Mundial et al., 2021). En consecuencia, la monetización *no es una valoración
primaria ni constituye por sí sola una cuenta de ecosistemas conforme al SCAE-CE*
(Naciones Unidas et al., 2021).

La homologación aplica el cociente entre los deflactores implícitos del PIB de 2026
(escenario bajo) y 2019, calculados como PIB nominal dividido por PIB real. Con las
series oficiales, el factor es 1.3296 (Banco de Guatemala, s. f.).

Se distinguen tres objetos: (a) flujo anual asociado a una cohorte anual de pérdida;
(b) valor presente de 25 años de servicios para esa cohorte; y (c) valor presente de
diez cohortes anuales. No deben intercambiarse como si fueran la misma magnitud.

Para un resultado acumulado $H_i$ de cuatro años, el flujo físico anual medio es

$$h_i=\frac{H_i}{4}.$$

Con valor unitario anual $v$, el flujo monetario es $F_i=v h_i$. Su valor presente
durante $T$ años a una tasa $r$ se calcula como

$$VP_i=F_i\left[\frac{1-(1+r)^{-T}}{r}\right].$$

Para diez cohortes anuales consecutivas:

$$VP_{i,10}=\sum_{k=1}^{10}\frac{VP_i}{(1+r)^k}.$$


In [21]:
#@title { display-mode: "form" }
valoracion = productos["valoracion_resultados_forestales_nacionales"].copy()
def intervalo_millones(fila, inferior, superior):
    bajo, alto = fila[inferior] / 1e6, fila[superior] / 1e6
    return f"{bajo:,.1f}" if np.isclose(bajo, alto) else f"{bajo:,.1f}–{alto:,.1f}"

tabla_valoracion = pd.DataFrame({
    "Resultado forestal": valoracion["regla"],
    "Flujo anual (Q millones)": valoracion.apply(
        intervalo_millones, axis=1,
        args=("flujo_anual_inferior_gtq", "flujo_anual_superior_gtq")
    ),
    "VP de una cohorte (Q millones)": valoracion.apply(
        intervalo_millones, axis=1,
        args=("vp_cohorte_inferior_gtq", "vp_cohorte_superior_gtq")
    ),
    "VP de 10 cohortes (Q millones)": valoracion.apply(
        intervalo_millones, axis=1,
        args=("vp_diez_cohortes_inferior_gtq", "vp_diez_cohortes_superior_gtq")
    ),
})

mostrar_tabla(
    tabla_valoracion,
    "Tabla 10. Valoración indicativa de los resultados forestales nacionales",
    "Q de 2026. Tasa central de 4%, horizonte de servicios de 25 años y diez cohortes anuales. Los intervalos solo difieren para el saldo ponderado.",
    FUENTE_VALORACION,
    decimales=1,
    max_filas=None,
    archivo="valoracion_resultados_forestales_guatemala_2016_2020_precios_2026.csv",
    descarga=valoracion,
)


Resultado forestal,Flujo anual (Q millones),VP de una cohorte (Q millones),VP de 10 cohortes (Q millones)
Deforestación bruta,"1,832.1","28,621.8","232,148.2"
Saldo ponderado por recuperación,873.2–929.5,"13,640.5–14,520.6","110,636.9–117,775.1"
Pérdida neta institucional,395.3,"6,176.1","50,093.9"


*Nota.* Q de 2026. Tasa central de 4%, horizonte de servicios de 25 años y diez cohortes anuales. Los intervalos solo difieren para el saldo ponderado.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

La valoración conserva el orden de los resultados físicos: la pérdida bruta produce el mayor monto, la pérdida neta el menor y el saldo ponderado un intervalo intermedio.


In [22]:
#@title { display-mode: "form" }
flujo = valoracion.copy()
flujo["central_millones"] = (
    flujo["flujo_anual_inferior_gtq"] + flujo["flujo_anual_superior_gtq"]
) / 2e6
flujo["err_mas"] = flujo["flujo_anual_superior_gtq"] / 1e6 - flujo["central_millones"]
flujo["err_menos"] = flujo["central_millones"] - flujo["flujo_anual_inferior_gtq"] / 1e6
fig_flujo = go.Figure(go.Bar(
    x=flujo["regla"], y=flujo["central_millones"],
    marker_color=["#D55E00", "#6A3D9A", "#0072B2"],
    error_y=dict(type="data", array=flujo["err_mas"], arrayminus=flujo["err_menos"]),
    text=[f"Q{v:,.0f} M" for v in flujo["central_millones"]], textposition="outside",
    hovertemplate="%{x}<br>Q%{y:,.1f} millones/año<extra></extra>",
))
maximo_flujo = (flujo["central_millones"] + flujo["err_mas"]).max()
fig_flujo.update_yaxes(
    title="Q millones por año", range=[0, maximo_flujo * 1.18], tickformat=","
)
fig_flujo.update_xaxes(title=None)

mostrar_figura(
    fig_flujo,
    "Figura 11. Flujo anual indicativo por resultado forestal",
    "La transferencia uniforme compara resultados, pero no representa valores municipales observados ni sustituye una valoración primaria.",
    FUENTE_VALORACION,
    alto=670,
)


Figura 11. Flujo anual indicativo por resultado forestal

*Nota.* La transferencia uniforme compara resultados, pero no representa valores municipales observados ni sustituye una valoración primaria.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

Como se aplica un valor unitario uniforme, las diferencias monetarias son proporcionales a las hectáreas anualizadas. La figura expresa órdenes de magnitud, no precios observados por municipio.


In [23]:
#@title { display-mode: "form" }
sensibilidad = productos["sensibilidad_valor_presente"].copy()
sensibilidad["VP inferior (Q millones)"] = sensibilidad["vp_cohorte_inferior_gtq"] / 1e6
sensibilidad["VP superior (Q millones)"] = sensibilidad["vp_cohorte_superior_gtq"] / 1e6
sensibilidad["VP de una cohorte (Q millones)"] = sensibilidad.apply(
    lambda f: (
        f"{f['VP inferior (Q millones)']:,.1f}"
        if np.isclose(f['VP inferior (Q millones)'], f['VP superior (Q millones)'])
        else f"{f['VP inferior (Q millones)']:,.1f}–{f['VP superior (Q millones)']:,.1f}"
    ), axis=1
)
tabla_sensibilidad = sensibilidad[["regla", "tasa", "VP de una cohorte (Q millones)"]].rename(columns={
    "regla": "Resultado forestal", "tasa": "Tasa de descuento"
})
tabla_sensibilidad["Tasa de descuento"] = tabla_sensibilidad["Tasa de descuento"].map(
    lambda x: f"{x:.0%}"
)

mostrar_tabla(
    tabla_sensibilidad,
    "Tabla 11. Sensibilidad del valor presente de una cohorte anual",
    "Horizonte de servicios de 25 años; tasas de 2%, 4% y 5%. El saldo ponderado conserva un intervalo.",
    FUENTE_VALORACION,
    decimales=2,
    max_filas=None,
    archivo="sensibilidad_valor_presente_guatemala_precios_2026.csv",
    descarga=sensibilidad,
)


Resultado forestal,Tasa de descuento,VP de una cohorte (Q millones)
Deforestación bruta,2%,"35,769.6"
Deforestación bruta,4%,"28,621.8"
Deforestación bruta,5%,"25,822.0"
Saldo ponderado por recuperación,2%,"17,047.0–18,146.9"
Saldo ponderado por recuperación,4%,"13,640.5–14,520.6"
Saldo ponderado por recuperación,5%,"12,306.2–13,100.2"
Pérdida neta institucional,2%,"7,718.5"
Pérdida neta institucional,4%,"6,176.1"
Pérdida neta institucional,5%,"5,572.0"


*Nota.* Horizonte de servicios de 25 años; tasas de 2%, 4% y 5%. El saldo ponderado conserva un intervalo.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

Una tasa de descuento menor eleva el valor presente de todas las alternativas. La sensibilidad financiera modifica los montos, pero no revierte su orden relativo.


In [24]:
#@title { display-mode: "form" }
sensibilidad["VP central (Q millones)"] = (
    sensibilidad["VP inferior (Q millones)"] + sensibilidad["VP superior (Q millones)"]
) / 2
colores_regla = {
    "Deforestación bruta": "#D55E00",
    "Saldo ponderado por recuperación": "#6A3D9A",
    "Pérdida neta institucional": "#0072B2",
}
trazos_regla = {
    "Deforestación bruta": "solid",
    "Saldo ponderado por recuperación": "dash",
    "Pérdida neta institucional": "dot",
}
simbolos_regla = {
    "Deforestación bruta": "circle",
    "Saldo ponderado por recuperación": "diamond",
    "Pérdida neta institucional": "square",
}
fig_sensibilidad = go.Figure()
for regla_nombre in [
    "Deforestación bruta", "Saldo ponderado por recuperación", "Pérdida neta institucional"
]:
    serie = sensibilidad.loc[sensibilidad["regla"].eq(regla_nombre)].sort_values("tasa")
    fig_sensibilidad.add_trace(go.Scatter(
        x=100 * serie["tasa"], y=serie["VP central (Q millones)"],
        mode="lines+markers", name=regla_nombre,
        line=dict(color=colores_regla[regla_nombre], dash=trazos_regla[regla_nombre], width=3),
        marker=dict(symbol=simbolos_regla[regla_nombre], size=9),
        error_y=dict(
            type="data",
            array=serie["VP superior (Q millones)"] - serie["VP central (Q millones)"],
            arrayminus=serie["VP central (Q millones)"] - serie["VP inferior (Q millones)"],
        ),
        hovertemplate="Tasa: %{x:.0f}%<br>VP: Q%{y:,.1f} millones<extra>%{fullData.name}</extra>",
    ))
fig_sensibilidad.update_xaxes(title="Tasa de descuento (%)", tickvals=[2, 4, 5])
fig_sensibilidad.update_yaxes(title="VP de una cohorte (Q millones de 2026)", tickformat=",")
mostrar_figura(
    fig_sensibilidad,
    "Figura 12. Sensibilidad del valor presente a la tasa de descuento",
    "Las líneas combinan color, trazo y marcador. El intervalo solo es visible para el saldo ponderado y el orden de los resultados no cambia entre tasas.",
    FUENTE_VALORACION,
    alto=680,
)


Figura 12. Sensibilidad del valor presente a la tasa de descuento

*Nota.* Las líneas combinan color, trazo y marcador. El intervalo solo es visible para el saldo ponderado y el orden de los resultados no cambia entre tasas.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

El intervalo ponderado permanece separado del resultado neto en las tres tasas. Por ello, la diferencia principal procede del tratamiento físico de la recuperación y no de una tasa particular.


## 6. Escenarios 2026–2035

Los escenarios modifican por separado la pérdida bruta y la recuperación mediante
$m_s^B$ y $m_s^R$:

$$H_{i,s}=m_s^B B_i-\rho_i m_s^R R_i.$$

Los tres escenarios son proporcionales: contención ($0.25,0.25$), continuidad
($1,1$) y deterioro acelerado ($2,2$). La contención *no se denomina restauración*,
porque no incorpora una trayectoria adicional de recuperación. La
formulación deja preparado el modelo para escenarios asimétricos futuros.


In [25]:
#@title { display-mode: "form" }
escenarios = productos["escenarios_nacionales"].copy()
tabla_escenarios = escenarios[[
    "escenario", "multiplicador_perdida_bruta", "multiplicador_recuperacion"
]].drop_duplicates().rename(columns={
    "escenario": "Escenario",
    "multiplicador_perdida_bruta": "Multiplicador de pérdida bruta",
    "multiplicador_recuperacion": "Multiplicador de recuperación",
}).sort_values("Escenario")

mostrar_tabla(
    tabla_escenarios,
    "Tabla 12. Multiplicadores físicos de los escenarios 2026–2035",
    "Los multiplicadores se aplican a los flujos base observados. Son supuestos comparativos, no pronósticos probabilísticos.",
    "Supuestos y cálculos del autor.",
    decimales=2,
    max_filas=None,
    archivo="escenarios_forestales_guatemala_2026_2035.csv",
)


Escenario,Multiplicador de pérdida bruta,Multiplicador de recuperación
Contención proporcional,0.25,0.25
Continuidad,1.00,1.00
Deterioro proporcional acelerado,2.00,2.00


*Nota.* Los multiplicadores se aplican a los flujos base observados. Son supuestos comparativos, no pronósticos probabilísticos.

*Fuente.* Supuestos y cálculos del autor.

Los escenarios alteran por separado pérdida y recuperación. Esa simetría hace explícito qué supuesto impulsa cada trayectoria y evita presentar los resultados como pronósticos.


In [26]:
#@title { display-mode: "form" }
trayectorias = productos["trayectorias_fisicas"].copy()
trayectorias["Acumulado central (ha)"] = (
    trayectorias["acumulado_inferior_ha"] + trayectorias["acumulado_superior_ha"]
) / 2
fig_trayectorias = px.line(
    trayectorias,
    x="anio", y="Acumulado central (ha)", color="regla", line_dash="regla",
    symbol="regla", markers=True, facet_col="escenario",
    facet_col_spacing=0.045,
    labels={"anio": "Año", "regla": "Resultado forestal", "escenario": "Escenario"},
    color_discrete_map={
        "Deforestación bruta": "#D55E00",
        "Saldo ponderado por recuperación": "#6A3D9A",
        "Pérdida neta institucional": "#0072B2",
    },
    line_dash_map={
        "Deforestación bruta": "solid",
        "Saldo ponderado por recuperación": "dash",
        "Pérdida neta institucional": "dot",
    },
    symbol_map={
        "Deforestación bruta": "circle",
        "Saldo ponderado por recuperación": "diamond",
        "Pérdida neta institucional": "square",
    },
)
fig_trayectorias.update_traces(line_width=3, marker_size=7)
fig_trayectorias.update_xaxes(title=None)
fig_trayectorias.update_yaxes(matches="y", title=None, tickformat=",")
fig_trayectorias.update_yaxes(title_text="ha acumuladas", row=1, col=1)
fig_trayectorias.update_layout(legend_title_text="Resultado forestal")
fig_trayectorias.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1].replace(
        "Deterioro proporcional acelerado", "Deterioro proporcional<br>acelerado"
    ))
)
fig_trayectorias.add_annotation(
    text="Año", x=0.5, y=0, xref="paper", yref="paper", yshift=-44,
    showarrow=False, font=dict(size=12, color="#24363D"),
)

mostrar_figura(
    fig_trayectorias,
    "Figura 13. Trayectorias físicas acumuladas bajo tres escenarios proporcionales",
    "Se grafica el punto medio del intervalo ponderado. Los paneles comparten escala vertical y los resultados se distinguen por color, trazo y marcador.",
    FUENTE_ESCENARIOS,
    alto=740,
)


Figura 13. Trayectorias físicas acumuladas bajo tres escenarios proporcionales

*Nota.* Se grafica el punto medio del intervalo ponderado. Los paneles comparten escala vertical y los resultados se distinguen por color, trazo y marcador.

*Fuente.* INAB y CONAP (2023) e INAB (2023b); supuestos y cálculos del autor.

La contención proporcional desacelera la acumulación; la continuidad conserva el ritmo base y el deterioro acelerado amplía la brecha. En los tres casos el saldo ponderado permanece entre la pérdida bruta y la pérdida neta.


In [27]:
#@title { display-mode: "form" }
trayectorias_monetarias = productos["trayectorias_monetarias"].copy()
trayectorias_monetarias["VP central (Q miles de millones)"] = (
    trayectorias_monetarias["vp_acumulado_inferior_gtq"]
    + trayectorias_monetarias["vp_acumulado_superior_gtq"]
) / 2e9
fig_trayectorias_monetarias = px.line(
    trayectorias_monetarias,
    x="anio", y="VP central (Q miles de millones)", color="regla", line_dash="regla",
    symbol="regla", markers=True, facet_col="escenario",
    facet_col_spacing=0.045,
    labels={"anio": "Año", "regla": "Resultado forestal", "escenario": "Escenario"},
    color_discrete_map={
        "Deforestación bruta": "#D55E00",
        "Saldo ponderado por recuperación": "#6A3D9A",
        "Pérdida neta institucional": "#0072B2",
    },
    line_dash_map={
        "Deforestación bruta": "solid",
        "Saldo ponderado por recuperación": "dash",
        "Pérdida neta institucional": "dot",
    },
    symbol_map={
        "Deforestación bruta": "circle",
        "Saldo ponderado por recuperación": "diamond",
        "Pérdida neta institucional": "square",
    },
)
fig_trayectorias_monetarias.update_traces(line_width=3, marker_size=7)
fig_trayectorias_monetarias.update_xaxes(title=None)
fig_trayectorias_monetarias.update_yaxes(
    matches="y", title=None, tickformat=",.1f"
)
fig_trayectorias_monetarias.update_yaxes(
    title_text="Q miles de millones, valor presente", row=1, col=1
)
fig_trayectorias_monetarias.update_layout(legend_title_text="Resultado forestal")
fig_trayectorias_monetarias.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1].replace(
        "Deterioro proporcional acelerado", "Deterioro proporcional<br>acelerado"
    ))
)
fig_trayectorias_monetarias.add_annotation(
    text="Año", x=0.5, y=0, xref="paper", yref="paper", yshift=-44,
    showarrow=False, font=dict(size=12, color="#24363D"),
)

mostrar_figura(
    fig_trayectorias_monetarias,
    "Figura 14. Trayectorias del valor presente acumulado bajo tres escenarios",
    "Punto medio del intervalo, Q de 2026, tasa de 4% y 25 años de servicios por cohorte. Los paneles comparten escala y los resultados se distinguen por color, trazo y marcador.",
    FUENTE_VALORACION,
    alto=740,
)


Figura 14. Trayectorias del valor presente acumulado bajo tres escenarios

*Nota.* Punto medio del intervalo, Q de 2026, tasa de 4% y 25 años de servicios por cohorte. Los paneles comparten escala y los resultados se distinguen por color, trazo y marcador.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

La incorporación sucesiva de cohortes hace crecer el valor presente acumulado. La distancia entre trayectorias combina los supuestos físicos con el mismo esquema de valoración.


In [28]:
#@title { display-mode: "form" }
escenarios_val = productos["escenarios_valorados"].copy()
escenarios_val["Resultado físico, década (ha)"] = escenarios_val.apply(
    lambda f: (
        f"{f.resultado_inferior_decada_ha:,.0f}"
        if np.isclose(f.resultado_inferior_decada_ha, f.resultado_superior_decada_ha)
        else f"{f.resultado_inferior_decada_ha:,.0f}–{f.resultado_superior_decada_ha:,.0f}"
    ), axis=1
)
escenarios_val["VP, década (Q millones)"] = escenarios_val.apply(
    lambda f: (
        f"{f.vp_decada_inferior_gtq / 1e6:,.1f}"
        if np.isclose(f.vp_decada_inferior_gtq, f.vp_decada_superior_gtq)
        else f"{f.vp_decada_inferior_gtq / 1e6:,.1f}–{f.vp_decada_superior_gtq / 1e6:,.1f}"
    ), axis=1
)
tabla_escenarios_val = escenarios_val[[
    "escenario", "regla", "Resultado físico, década (ha)", "VP, década (Q millones)"
]].rename(columns={"escenario": "Escenario", "regla": "Resultado forestal"}).sort_values(["Escenario", "Resultado forestal"])

mostrar_tabla(
    tabla_escenarios_val,
    "Tabla 13. Resultados físicos y valoración indicativa de los escenarios",
    "Valor presente de diez cohortes anuales, Q de 2026, tasa de 4% y 25 años de servicios por cohorte. Los resultados son alternativos y no se suman.",
    FUENTE_VALORACION,
    decimales=1,
    max_filas=None,
    archivo="escenarios_forestales_valorados_guatemala_2026_2035.csv",
    descarga=escenarios_val,
)


Escenario,Resultado forestal,"Resultado físico, década (ha)","VP, década (Q millones)"
Contención proporcional,Deforestación bruta,"152,747","58,037.0"
Contención proporcional,Pérdida neta institucional,"32,960","12,523.5"
Contención proporcional,Saldo ponderado por recuperación,"72,796–77,493","27,659.2–29,443.8"
Continuidad,Deforestación bruta,"610,986","232,148.2"
Continuidad,Pérdida neta institucional,"131,841","50,093.9"
Continuidad,Saldo ponderado por recuperación,"291,183–309,970","110,636.9–117,775.1"
Deterioro proporcional acelerado,Deforestación bruta,"1,221,973","464,296.4"
Deterioro proporcional acelerado,Pérdida neta institucional,"263,682","100,187.7"
Deterioro proporcional acelerado,Saldo ponderado por recuperación,"582,366–619,940","221,273.7–235,550.2"


*Nota.* Valor presente de diez cohortes anuales, Q de 2026, tasa de 4% y 25 años de servicios por cohorte. Los resultados son alternativos y no se suman.

*Fuente.* Banco Mundial et al. (2021) y Banco de Guatemala (s. f.); cálculos del autor.

La tabla permite comparar, bajo supuestos comunes, el efecto conjunto del escenario físico y del resultado forestal utilizado como base de la valoración.


## 7. Manglar: aproximación local con evidencia estructural

El módulo local parte de 55 trayectorias multitemporales de estructura de manglar
disponibles en el portal de INAB: 30 favorables, 21 desfavorables y 4 mixtas. De ahí
se deriva $\omega_M$, el *ponderador estructural local de manglar*:

$$\underline{\omega}_M=\frac{n_F}{n}=\frac{30}{55}=0.5455,$$

$$\overline{\omega}_M=\frac{n_F+n_M}{n}=\frac{34}{55}=0.6182.$$

El límite inferior cuenta solo trayectorias con aumento conjunto de carbono y área
basal; el superior incorpora las cuatro trayectorias mixtas. No es un intervalo de
confianza. Las 55 series representan 73.3 % de los 75 registros de parcela contenidos
en el archivo analítico; esa cobertura se documenta y no se usa como multiplicador.

Para cada municipio, la aplicación se define como

$$H_{i,M}(\omega_M)=B_i-\omega_MR_i,$$

$$\underline H_{i,M}=B_i-\overline{\omega}_MR_i,\qquad
\overline H_{i,M}=B_i-\underline{\omega}_MR_i.$$

La inversión de los extremos se debe a que $\partial H_{i,M}/\partial\omega_M=-R_i$.
Con la tolerancia numérica $\varepsilon=10^{-8}$ utilizada por el código, la
clasificación es pérdida si $\underline H_{i,M}>\varepsilon$, ganancia si
$\overline H_{i,M}<-\varepsilon$ e indeterminada en otro caso. Además,

$$H_{i,M}(\omega_M)-N_i=(1-\omega_M)R_i.$$

$B_i$ y $R_i$ siguen siendo la pérdida y la ganancia de cobertura forestal del
municipio completo, no cambios exclusivos de manglar. La evidencia de campo informa
únicamente el ponderador local. Los resultados se comparan con el cálculo neto y,
donde existe soporte común, con la recuperación ponderada a veinte años; no se suman
entre sí (INAB et al., 2016; INAB, 2023; Poorter et al., 2016, 2017).


In [29]:
#@title { display-mode: "form" }
intervalo_local = productos["intervalo_estructural_local"].copy().rename(columns={
    "series_multitemporales": "Series multitemporales",
    "trayectorias_favorables": "Favorables",
    "trayectorias_desfavorables": "Desfavorables",
    "trayectorias_mixtas": "Mixtas",
    "proporcion_estructural_min": "Proporción estructural mínima",
    "proporcion_estructural_max": "Proporción estructural máxima",
})
intervalo_local_visible = pd.DataFrame({
    "Series": intervalo_local["Series multitemporales"],
    "Trayectorias (F/D/M)": intervalo_local.apply(
        lambda f: f"{int(f.Favorables)} / {int(f.Desfavorables)} / {int(f.Mixtas)}", axis=1
    ),
    "Proporción mínima": intervalo_local["Proporción estructural mínima"],
    "Proporción máxima": intervalo_local["Proporción estructural máxima"],
})

mostrar_tabla(
    intervalo_local_visible,
    "Tabla 14. Derivación del intervalo estructural local de manglar",
    "El límite inferior cuenta solo trayectorias favorables; el superior incorpora las cuatro mixtas. No es una proporción nacional de recuperación de manglar.",
    "INAB (2023a) e INAB et al. (2016); cálculos del autor.",
    decimales=3,
    max_filas=None,
    archivo="intervalo_estructural_manglar_guatemala.csv",
    descarga=productos["intervalo_estructural_local"],
)


Series,Trayectorias (F/D/M),Proporción mínima,Proporción máxima
55.000,30 / 21 / 4,0.545,0.618


*Nota.* El límite inferior cuenta solo trayectorias favorables; el superior incorpora las cuatro mixtas. No es una proporción nacional de recuperación de manglar.

*Fuente.* INAB (2023a) e INAB et al. (2016); cálculos del autor.

Treinta de las 55 series muestran aumento conjunto de carbono y área basal; cuatro son mixtas. De esos conteos se deriva un ponderador estructural local de 0.545–0.618.


In [30]:
#@title { display-mode: "form" }
evidencia_mangle = productos["evidencia_manglar"].copy()
evidencia_mangle["Total"] = evidencia_mangle["series_multitemporales"]
orden_mangle = evidencia_mangle.sort_values(["Total", "municipio"])["municipio"].tolist()
categorias_mangle = [
    ("Aumento conjunto", "suben_carbono_y_area_basal", "#009E73"),
    ("Disminución conjunta", "bajan_carbono_y_area_basal", "#D55E00"),
    ("Mixta", "trayectoria_mixta", "#6A3D9A"),
]
fig_evidencia_mangle = go.Figure()
for etiqueta, columna, color in categorias_mangle:
    valores = evidencia_mangle[columna]
    fig_evidencia_mangle.add_trace(go.Bar(
        x=valores,
        y=evidencia_mangle["municipio"],
        orientation="h",
        name=etiqueta,
        marker_color=color,
        text=[str(int(v)) if v > 0 else "" for v in valores],
        textposition="inside",
        insidetextanchor="middle",
        hovertemplate="%{y}<br>%{fullData.name}: %{x:.0f}<extra></extra>",
    ))
fig_evidencia_mangle.update_layout(
    barmode="stack", legend_title_text="Trayectoria estructural",
    margin=dict(l=210, r=60),
)
fig_evidencia_mangle.update_xaxes(
    title="Número de series multitemporales", range=[0, 17.5], dtick=2
)
fig_evidencia_mangle.update_yaxes(
    title=None, categoryorder="array", categoryarray=orden_mangle
)
fig_evidencia_mangle.add_annotation(
    x=0.25, y="Nueva Concepción", text="0", showarrow=False,
    xanchor="left", font=dict(color="#5C6F77", size=11),
)

mostrar_figura(
    fig_evidencia_mangle,
    "Figura 15. Distribución municipal de las series estructurales de manglar",
    "Las categorías describen el cambio conjunto de carbono y área basal en las series multitemporales. Nueva Concepción tiene cinco registros de parcela, pero ninguna serie multitemporal disponible; el ponderador utiliza las 55 series clasificadas.",
    "INAB (2023a), Áreas potenciales de restauración de manglares; cálculos del autor.",
    alto=680,
)


Figura 15. Distribución municipal de las series estructurales de manglar

*Nota.* Las categorías describen el cambio conjunto de carbono y área basal en las series multitemporales. Nueva Concepción tiene cinco registros de parcela, pero ninguna serie multitemporal disponible; el ponderador utiliza las 55 series clasificadas.

*Fuente.* INAB (2023a), Áreas potenciales de restauración de manglares; cálculos del autor.

La Blanca y Retalhuleu reúnen 29 series, equivalentes al 52.7 % del total. Al sumar Sipacate y Pasaco, cuatro municipios concentran 43 de las 55 series, o 78.2 % de la evidencia clasificada.


In [31]:
#@title { display-mode: "form" }
resumen_local = productos["resumen_mangle_local"].iloc[0]
tabla_resumen_local = pd.DataFrame({
    "Magnitud": [
        "Municipios", "Pérdida bruta", "Ganancia de cobertura",
        "Pérdida neta", "Saldo con ponderador estructural",
    ],
    "Resultado": [
        f"{int(resumen_local.municipios)}",
        f"{resumen_local.perdida_bruta_ha:,.1f} ha",
        f"{resumen_local.recuperacion_bruta_ha:,.1f} ha",
        f"{resumen_local.perdida_neta_ha:,.1f} ha",
        f"{resumen_local.saldo_estructural_inferior_ha:,.1f}–{resumen_local.saldo_estructural_superior_ha:,.1f} ha",
    ],
    "Lectura": ["Ámbito local", "B", "R", "N = B − R", "H_M"],
})

mostrar_tabla(
    tabla_resumen_local,
    "Tabla 15. Resultados agregados en trece municipios con evidencia de manglar",
    "La pérdida y recuperación proceden de la base municipal general; la evidencia de campo sustenta únicamente la ponderación estructural local.",
    FUENTE_MANGLE,
    decimales=1,
    max_filas=None,
    archivo="resumen_manglar_guatemala_2016_2020.csv",
    descarga=productos["resumen_mangle_local"],
)


Magnitud,Resultado,Lectura
Municipios,13,Ámbito local
Pérdida bruta,"12,990.4 ha",B
Ganancia de cobertura,"7,950.8 ha",R
Pérdida neta,"5,039.6 ha",N = B − R
Saldo con ponderador estructural,"8,075.3–8,653.6 ha",H_M


*Nota.* La pérdida y recuperación proceden de la base municipal general; la evidencia de campo sustenta únicamente la ponderación estructural local.

*Fuente.* INAB (2023a), INAB et al. (2016), e INAB y CONAP (2023); cálculos del autor.

El saldo agregado aumenta de 5,039.6 ha bajo el cálculo neto a 8,075.3–8,653.6 ha con el ponderador estructural. La diferencia es de 3,035.7–3,614.0 ha, equivalente a un resultado 60.2 %–71.7 % mayor que la pérdida neta.


In [32]:
#@title { display-mode: "form" }
local_comp = productos["comparacion_recuperacion_ponderada_mangle"].copy()
graf_local = local_comp[[
    "municipio", "perdida_neta_ha",
    "saldo_estructural_inferior_ha", "saldo_estructural_superior_ha",
    "saldo_ponderado_inferior_ha", "saldo_ponderado_superior_ha"
]].sort_values("saldo_estructural_superior_ha")
fig_local = go.Figure()
fig_local.add_trace(go.Scatter(
    x=graf_local["perdida_neta_ha"], y=graf_local["municipio"],
    mode="markers", name="Pérdida neta", marker=dict(color="#0072B2", size=9, symbol="square"),
    hovertemplate="%{y}<br>Pérdida neta: %{x:,.1f} ha<extra></extra>",
))
for etiqueta, inferior, superior, color, simbolo in [
    ("Aproximación estructural local", "saldo_estructural_inferior_ha", "saldo_estructural_superior_ha", "#009E73", "diamond"),
    ("Saldo ponderado con ρ₂₀", "saldo_ponderado_inferior_ha", "saldo_ponderado_superior_ha", "#E69F00", "circle"),
]:
    centro = (graf_local[inferior] + graf_local[superior]) / 2
    fig_local.add_trace(go.Scatter(
        x=centro, y=graf_local["municipio"], mode="markers", name=etiqueta,
        marker=dict(color=color, size=9, symbol=simbolo),
        error_x=dict(type="data", array=graf_local[superior]-centro, arrayminus=centro-graf_local[inferior]),
        hovertemplate="%{y}<br>%{x:,.1f} ha<extra></extra>",
    ))
fig_local.update_xaxes(title="Resultado (ha); pérdida positiva", zeroline=True, tickformat=",")
fig_local.update_yaxes(title=None)

mostrar_figura(
    fig_local,
    "Figura 16. Resultados locales en municipios con evidencia de manglar",
    "Los puntos e intervalos son resultados alternativos de la misma pérdida y ganancia de cobertura municipal. La aplicación estructural local y la recuperación ponderada a veinte años no se agregan entre sí.",
    "INAB y CONAP (2023), INAB (2023a) y Poorter et al. (2016, 2017); cálculos del autor.",
    alto=760,
)


Figura 16. Resultados locales en municipios con evidencia de manglar

*Nota.* Los puntos e intervalos son resultados alternativos de la misma pérdida y ganancia de cobertura municipal. La aplicación estructural local y la recuperación ponderada a veinte años no se agregan entre sí.

*Fuente.* INAB y CONAP (2023), INAB (2023a) y Poorter et al. (2016, 2017); cálculos del autor.

El resultado neto clasifica seis municipios con pérdida y siete con ganancia. El ponderador local clasifica diez con pérdida y tres con ganancia; Tiquisate, Chiquimulilla, La Blanca y Pasaco cambian de signo.


In [33]:
#@title { display-mode: "form" }
tabla_local = local_comp[[
    "depto", "municipio", "perdida_neta_ha"
]].copy()
tabla_local["Intervalo estructural (ha)"] = local_comp.apply(
    lambda f: f"{f.saldo_estructural_inferior_ha:,.1f}–{f.saldo_estructural_superior_ha:,.1f}", axis=1
)
tabla_local["Saldo ponderado con ρ₂₀ (ha)"] = local_comp.apply(
    lambda f: f"{f.saldo_ponderado_inferior_ha:,.1f}–{f.saldo_ponderado_superior_ha:,.1f}", axis=1
)
tabla_local = tabla_local.rename(columns={
    "depto": "Departamento", "municipio": "Municipio",
    "perdida_neta_ha": "Pérdida neta (ha)",
}).sort_values(["Departamento", "Municipio"])

mostrar_tabla(
    tabla_local,
    "Tabla 16. Comparación municipal de la aproximación estructural local",
    "Los trece municipios tienen soporte común para la comparación. Los intervalos estructural y de recuperación ponderada responden a fundamentos distintos y no son componentes aditivos.",
    "INAB y CONAP (2023), INAB (2023a) y Poorter et al. (2016, 2017); cálculos del autor.",
    decimales=1,
    max_filas=None,
    archivo="comparacion_recuperacion_ponderada_y_manglar_municipios_guatemala_2016_2020.csv",
    descarga=local_comp,
)


Departamento,Municipio,Pérdida neta (ha),Intervalo estructural (ha),Saldo ponderado con ρ₂₀ (ha)
Escuintla,Iztapa,0.5,47.1–56.0,29.1–50.2
Escuintla,Nueva Concepción,64.3,115.1–124.8,95.4–118.5
Escuintla,Sipacate,613.5,664.5–674.2,644.8–667.9
Escuintla,Tiquisate,-17.5,69.1–85.6,35.6–74.8
Izabal,Livingston,"5,099.6","5,887.6–6,037.8","5,582.6–5,939.6"
Izabal,Puerto Barrios,"1,915.3","1,951.2–1,958.1","1,937.3–1,953.6"
Jutiapa,Pasaco,-18.6,13.3–19.4,-6.0–37.0
Retalhuleu,Champerico,-607.5,-221.5–-148.0,-370.9–-196.0
Retalhuleu,Retalhuleu,"-1,115.1",-316.3–-164.1,-625.5–-263.6
San Marcos,La Blanca,-28.0,23.0–32.7,3.2–26.3


*Nota.* Los trece municipios tienen soporte común para la comparación. Los intervalos estructural y de recuperación ponderada responden a fundamentos distintos y no son componentes aditivos.

*Fuente.* INAB y CONAP (2023), INAB (2023a) y Poorter et al. (2016, 2017); cálculos del autor.

Los intervalos estructural y de recuperación ponderada se superponen en el agregado, pero responden a fundamentos distintos. En doce municipios conservan la misma clasificación; Pasaco es pérdida en la aplicación local e indeterminado con la proporción de recuperación a veinte años.


## 8. Recuadro de desastres y degradación: contexto no aditivo

Los costos reportados para degradación de suelos y para las tormentas Eta e Iota
ayudan a dimensionar el entorno económico de la pérdida de capital natural. No se
atribuyen causalmente a la deforestación de este ejercicio y no se suman a la
valoración forestal: hacerlo produciría doble conteo y una precisión inexistente
(Castañeda Sánchez et al., 2019; CEPAL, 2021).


In [34]:
#@title { display-mode: "form" }
costos = productos["costos_contextuales"].copy().rename(columns={
    "dimension": "Dimensión",
    "indicador_fuente": "Indicador",
    "equivalencia_indicativa_2026_gtq_millones": "Equivalencia 2026 (Q millones)",
    "fuente": "Fuente original",
    "uso_analitico": "Uso analítico",
})
costos_visibles = costos[["Dimensión", "Indicador", "Equivalencia 2026 (Q millones)"]]

mostrar_tabla(
    costos_visibles,
    "Recuadro 1. Costos ambientales y de desastres presentados como contexto",
    "Las equivalencias a 2026 son indicativas. No se establece causalidad con la pérdida forestal ni se agregan estos montos a la valoración de servicios ecosistémicos.",
    "Castañeda Sánchez et al. (2019), CEPAL (2021) y Banco de Guatemala (s. f.); cálculos del autor.",
    decimales=0,
    max_filas=None,
    archivo="costos_contextuales_no_aditivos_guatemala_2026.csv",
    descarga=costos,
)


Dimensión,Indicador,Equivalencia 2026 (Q millones)
Degradacion de suelos y tierras,0.55 por ciento del PIB anual,"5,544"
Eta e Iota,0.1 puntos porcentuales del crecimiento del PIB,"1,008"


*Nota.* Las equivalencias a 2026 son indicativas. No se establece causalidad con la pérdida forestal ni se agregan estos montos a la valoración de servicios ecosistémicos.

*Fuente.* Castañeda Sánchez et al. (2019), CEPAL (2021) y Banco de Guatemala (s. f.); cálculos del autor.

Las cifras muestran la escala económica del contexto ambiental y de desastres. Se mantienen fuera de la valoración forestal porque la información disponible no permite atribuirlas a la deforestación ni descartar doble conteo.


## Descargas

El paquete integral reúne las tablas completas, el manifiesto de archivos, los
metadatos de ejecución y las instrucciones de citación. Los controles de consistencia
se ejecutan en el proceso reproducible y permanecen disponibles en el repositorio,
sin ocupar un apartado de resultados en el cuaderno.


In [35]:
#@title { display-mode: "form" }
archivos_descarga = [
    productos["zip"],
    repo / "05_verificacion" / "manifiesto_resultados.csv",
    repo / "05_verificacion" / "metadatos_ejecucion.json",
    repo / "como_citar.txt",
]
panel_descargas(
    archivos_descarga,
    titulo="Descargas reproducibles: tablas, manifiesto y metadatos",
)


Resultado HTML reproducible

## Conclusión

La comparación muestra que la pérdida neta puede disminuir mientras la pérdida bruta
permanece elevada, porque el cálculo institucional descuenta la recuperación de forma
completa dentro del mismo período. Incluso al conceder un horizonte de veinte años,
el saldo ponderado conserva una pérdida mayor que el resultado neto. La aproximación
cuantitativa no sustituye una medición ecológica longitudinal; muestra qué puede
establecerse al vincular fuentes disponibles y qué preguntas requieren investigación
adicional.


## Referencias

Banco de Guatemala. (s. f.). *Producto interno bruto total, año de referencia 2013:
Años 2013–2026* [Cuadro estadístico]. Recuperado el 26 de agosto de 2026, de
https://banguat.gob.gt/sites/default/files/banguat/cuentasnac/PIB2013/resumidos/1.1_PIB_Tasa_de_Variacion_AR2013.pdf

Banco Mundial, Gobierno de Guatemala, Alianza Mundial para la Contabilidad de la
Riqueza y la Valoración de los Servicios de los Ecosistemas, & Universidad Rafael
Landívar, Vicerrectoría de Investigación y Proyección. (2021). *Cuenta de ecosistemas
de Guatemala* (2.ª ed.). Universidad Rafael Landívar. https://documents.worldbank.org/en/publication/documents-reports/documentdetail/451591561110110128

Castañeda Sánchez, J. P., Carrera, J., & Rexhepi, D. (2019). *Towards natural capital
accounting in Guatemala: Synthesis report*. World Bank. https://documents1.worldbank.org/curated/en/332151561104488571/pdf/Towards-Natural-Capital-Accounting-in-Guatemala-Synthesis-Report.pdf

Comisión Económica para América Latina y el Caribe. (2021). *Evaluación de los
efectos e impactos de las depresiones tropicales Eta y Iota en Guatemala*
(LC/TS.2021/21). https://www.cepal.org/es/publicaciones/46681-evaluacion-efectos-impactos-depresiones-tropicales-eta-iota-guatemala

Instituto Nacional de Bosques. (2023a). *Áreas potenciales de restauración de
manglares* [Portal de información de campo]. https://sig.inab.gob.gt/portal/apps/storymaps/stories/955793375059405ab4964bb40813b9fd

Instituto Nacional de Bosques. (2023b). *Dinámica de la cobertura forestal
2016–2020: Tabla municipal* [Capa ArcGIS]. https://sig.inab.gob.gt/portal/home/item.html?id=a15d600e7aed41d8b2afdcdcefad32db&sublayer=5

Instituto Nacional de Bosques, & Consejo Nacional de Áreas Protegidas. (2023).
*Estudio de la cobertura forestal para el año 2020 y dinámica de la cobertura
forestal en el período 2016–2020: República de Guatemala* [Informe técnico].
https://sig.inab.gob.gt/portal/apps/storymaps/stories/eac535d7b61a47f7b12a9b81eb9c15b6

Instituto Nacional de Bosques, Instituto Privado de Investigación sobre Cambio
Climático, & Consejo Nacional de Áreas Protegidas. (2016). *Metodología para el
establecimiento y mantenimiento de parcelas permanentes de medición forestal (PPMF)
en bosque natural del ecosistema manglar*. https://icc.org.gt/wp-content/uploads/2023/03/094.pdf

Naciones Unidas, Comisión Europea, Organización de las Naciones Unidas para la
Alimentación y la Agricultura, Fondo Monetario Internacional, Organización para la
Cooperación y el Desarrollo Económicos, & Banco Mundial. (2021). *Sistema de
Contabilidad Ambiental y Económica—Contabilidad de los Ecosistemas (SCAE-CE)*.
https://seea.un.org/content/system-environmental-economic-accounting-ecosystem-accounting-white-cover-version

Poorter, L., Bongers, F., Aide, T. M., Almeyda Zambrano, A. M., Balvanera, P.,
Becknell, J. M., Boukili, V., Brancalion, P. H. S., Broadbent, E. N., Chazdon, R. L.,
Craven, D., de Almeida-Cortez, J. S., Cabral, G. A. L., de Jong, B. H. J., Denslow,
J. S., Dent, D. H., DeWalt, S. J., Dupuy, J. M., Durán, S. M., ... Rozendaal, D. M. A.
(2016). Biomass resilience of Neotropical secondary forests. *Nature, 530*, 211–214.
https://doi.org/10.1038/nature16512

Poorter, L., Bongers, F., Aide, T. M., Almeyda Zambrano, A. M., Balvanera, P.,
Becknell, J. M., Boukili, V., Brancalion, P. H. S., Broadbent, E. N., Chazdon, R. L.,
Craven, D., de Almeida-Cortez, J. S., Cabral, G. A. L., de Jong, B. H. J., Denslow,
J. S., Dent, D. H., DeWalt, S. J., Dupuy, J. M., Durán, S. M., ... Rozendaal, D. M. A.
(2017). *Data from: Biomass resilience of Neotropical secondary forests* [Data set].
Dryad. https://doi.org/10.5061/dryad.82vr4

Sandoval García, C. A., Gálvez Ruano, J. J., & Pinillos Cifuentes, D. A. (2022).
*Bosques*. Universidad Rafael Landívar, Editorial Cara Parens. https://biblior.url.edu.gt/wp-content/uploads/publichlg/IARNA/serie_ambi/978-9929-54-422-2.pdf


## Cómo citar

> Osorio, J. A. (2026). *Deforestación bruta, recuperación y saldo forestal
> ponderado en Guatemala* (Versión 1.0.0) [Cuaderno reproducible]. Instituto de
> Investigación en Ciencias Naturales y Tecnología, Universidad Rafael Landívar.
> https://doi.org/10.5281/zenodo.22119075

Esta referencia identifica la versión pública 1.0.0. Cada figura y tabla está
acompañada por la atribución de sus fuentes primarias.
